# GameTheory-09c : Stackelberg Security Game — patrouille, capteur imparfait, signaling

**Objectifs** : calculer une meilleure réponse SSE, comparer quatre politiques et mesurer la sensibilité.

**Prérequis** : espérance, jeux de Stackelberg et C#. **Durée** : 75 minutes.

**Navigation** : [Version Python](GameTheory-09c-Stackelberg-SecurityGame-Python.ipynb) | [GameTheory-09b-Commitment-Stackelberg-Python](./GameTheory-09b-Commitment-Stackelberg-Python.ipynb) — engagement crédible · [GameTheory-09d-Stackelberg-PPO-Co-Design-Python](./GameTheory-09d-Stackelberg-PPO-Co-Design-Python.ipynb) — Stackelberg appris par gradient, co-design · [Sommaire GameTheory](./README.md)

## Pourquoi ce notebook

GT-9b a montré qu'**engager** une action sous-optimale peut être l'optimum du leader dans un Stackelberg. Mais GT-9b se limite à une cible unique : l'incumbent choisit **une** action et l'entrant **une** réponse.

En *sécurité* (aéroports, parcs, réseaux), le défenseur alloue des **ressources limitées** sur **plusieurs cibles**, et l'attaquant choisit **la cible à frapper** après avoir observé le déploiement. Le bon modèle est le **Stackelberg security game** (Tambe 2011, Bondi et al. 2019). Deux complications ajoutées par Bondi et al. :

1. **Capteur imparfait** : faux négatifs (probabilité `p_fn` qu'une attaque réelle passe inaperçue).
2. **Signaling** : le défenseur peut **réagir** à un signal (alerte), avec un coût de réaction ; ignorer le signal est possible.

Le source Python situe cette comparaison dans le contexte bibliographique des jeux de sécurité. Le présent calcul compare les quatre politiques définies ci-dessous ; il ne reproduit pas une instance d'une publication extérieure.

## Ce qu'on construit

Une instance Stackelberg security game avec :
- un graphe de cibles `T` (3 cibles, valeurs hétérogènes) ;
- `R` ressources défenseur à allouer (patrouilles) ;
- un capteur imparfait (probabilité de détection `p_d` ; faux négatifs `p_fn`) ;
- une réaction optionnelle du défenseur à l'alerte (coût + bénéfice) ;
- paiements défenseur/attaquant pour chaque cible (succès/échec attaque, succès/échec défense).

Solveur exact : pour la petite instance (3 cibles), **énumération** des allocations (`C(3,0) + C(3,1) + C(3,2) = 7` cas) + meilleure réponse du follower (tie-breaking SSE = Strong-Stackelberg Equilibrium).

Comparaisons (4 stratégies) :
1. **NO-RES** : pas de déploiement (baseline).
2. **DET-SDP** : SDP classique (Tambe 2011), ignorant l'incertitude du capteur.
3. **UNC-PESS** : variante **pessimiste** (utilise `react=False` partout).
4. **UNC-OPT** : intègre l'incertitude (réaction optimale espérée).

Sweep `p_fn` (taux de faux négatifs) : montre **quand** le capteur aide ou nuit.

## Contexte bibliographique hérité du source

- **Tambe 2011**, *Security and Game Theory: Algorithms, Deployed Systems, Lessons Learned*, Cambridge UP — §2 (Stackelberg security game de base, sans incertitude).
- **Bondi, Oh, Baker, Albert & Sintov 2019**, *Exploiting Uncertain Real-Time Information from Deep Learning in Signaling Games for Security and Sustainability*, SGO paper 27 — §4 (instance, faux négatifs, signaling, résultat "pire que zéro drone").

**Raccord pédagogique GT-9b** : on prolonge le *leader engage, follower observe* en *leader alloue sur un graphe, follower choisit la cible après observation*, et on garde le vocabulaire de l'engagement crédible — la stratégie du défenseur est *annoncée* (commit), *pas révocable* (le follower la voit, contrairement à l'**Exercice 3** de GT-9b, « l'évanouissement mesuré — annonce révocable et caution minimale », où l'annonce, elle, était révocable).

**Référence pédagogique primaire** : [source Python versionné](https://github.com/jsboige/CoursIA/blob/a8566836bb4246cfedbf6923f63fb0b2ab37d368/MyIA.AI.Notebooks/GameTheory/GameTheory-09c-Stackelberg-SecurityGame-Python.ipynb). Les attributions et notices ci-dessus sont le contexte hérité de ce source ; elles ne constituent pas des lectures ou une vérification de première main des publications par ce jumeau. Les résultats revendiqués sont ceux de son modèle explicite et des sorties affichées.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
CultureInfo.DefaultThreadCurrentCulture=CultureInfo.InvariantCulture;
Console.WriteLine($"Runtime : {System.Runtime.InteropServices.RuntimeInformation.FrameworkDescription} ; énumération exacte BCL, sans solveur externe");

Runtime : .NET 10.0.14 ; énumération exacte BCL, sans solveur externe


## 1. Instance du jeu — graphe de cibles, paiements, capteur, réaction

Trois cibles `T0, T1, T2` de valeurs défense `v_d` et attaque `v_a` hétérogènes. La cible `T1` est **plus précieuse** pour l'attaquant que `T0` (paradoxe classique des SSG). Le défenseur dispose de `R=2` unités de patrouille, chaque cible nécessitant 0 ou 1 patrouille pour être *couverte*.

**Capteur** : un signal `s ∈ {0, 1}` est observé par le défenseur après l'attaque.
- Avec probabilité `p_d` (vrai positif) le capteur détecte correctement une attaque qui *touche* une cible couverte.
- Avec probabilité `p_fn = 1 - p_d` (faux négatif) le capteur *rate* une attaque même couverte.
- Pas de faux positifs ici : le modèle exécuté pose `p_fp = 0`. Cette hypothèse aura une conséquence structurelle sur le verdict §3 : elle exclut les réactions coûteuses déclenchées sans attaque ; le résultat de §3 dépend aussi de l'allocation vide admissible.

**Réaction** : si le capteur sonne (`s=1`), le défenseur peut *réagir* — mobiliser une 3ᵉ unité qui **intercepte** l'attaque. La sémantique exacte :
- l'attaquant est pris : son paiement chute de `b_react` en plus de sa pénalité de couverture ;
- le défenseur **sauve la cible** : il ne subit plus `V_D(t)`, il ne paie que le coût de mobilisation `c_react`.

Réagir n'est **pas toujours rationnel** : sur une cible de faible valeur, `c_react` dépasse la perte évitée — la décision appartient au défenseur, cible par cible, **après** le signal (elle est calculée en §1, pas supposée). Sans signal (faux négatif ou cible non couverte), pas de réaction possible.

**Paiements** (réalisés, hors aléa du capteur — l'espérance est prise au §2) :
- Attaquant choisit `t ∈ T` → `U_A = V_A(t)`, défenseur subit `U_D = V_D(t)`.
- Couverture de `t` (patrouille **physiquement présente**, que le capteur sonne ou non) : `U_D = V_D(t) + cover_bonus` ; `U_A = V_A(t) − cover_penalty`.
- Réaction (interception) : `U_A = V_A(t) − cover_penalty − b_react` ; `U_D = −c_react`.

In [2]:
string[] targets={"T0","T1","T2"};
var vd=new Dictionary<string,double>{{"T0",-1},{"T1",-2.5},{"T2",-4}};
var va=new Dictionary<string,double>{{"T0",3},{"T1",4},{"T2",2}};
int R=2;double coverBonus=1.5,coverPenalty=1,pD=.7,cReact=.5,bReact=2;
Console.WriteLine($"Cibles {string.Join(", ",targets)} ; R={R}, p_d={pD:F2}, p_fn={1-pD:F2}, coût réaction={cReact:F2}, pénalité={bReact:F2}");
foreach(var t in targets) Console.WriteLine($"{t} : V_D={vd[t]:F2}, V_A={va[t]:F2}");

Cibles T0, T1, T2 ; R=2, p_d=0.70, p_fn=0.30, coût réaction=0.50, pénalité=2.00


T0 : V_D=-1.00, V_A=3.00


T1 : V_D=-2.50, V_A=4.00


T2 : V_D=-4.00, V_A=2.00


### Lecture : le paradoxe de la cible précieuse

Notez `V_A[T1] = 4 > V_A[T0] = 3 > V_A[T2] = 2`, alors que l'ordre des pertes défenseur est l'inverse : `|V_D[T0]| = 1.0 < |V_D[T1]| = 2.5 < |V_D[T2]| = 4.0`. Autrement dit `V_D[T0] = -1` (peu coûteux à perdre), `V_D[T1] = -2.5`, `V_D[T2] = -4` (très coûteux à perdre) : les préférences divergent : T1 est la plus attractive pour l'attaquant tandis que T2 coûte le plus au défenseur — `T2` est la cible la plus chère au défenseur et la moins intéressante pour l'attaquant, `T1` l'inverse.

Le choix calculé dans ce modèle est le suivant : à l'optimum du §2, `T2` — la cible la plus précieuse — reste **non couverte** (allocation `['T0', 'T1']`, sortie du solveur), parce qu'une patrouille sur une cible que l'attaquant ne convoite pas est une unité perdue. Sans aucun déploiement, l'attaquant prend `T1`, sa cible la plus rentable (sortie `NO-RES` : `t* = T1`, `U_A = +4.00`).

Avec couverture : `V_D[T2] + COVER_BONUS = -4 + 1.5 = -2.5` (perte réduite si couvert). Pour l'attaquant : `V_A[T1] - COVER_PENALTY_A = 3` (toujours rentable).

In [3]:
double PayoffA(string t,bool covered,bool react)=>va[t]-(covered?coverPenalty:0)-(react?bReact:0);
double PayoffD(string t,bool covered,bool react)=>react?-cReact:vd[t]+(covered?coverBonus:0);
bool Reaction(string t)=>PayoffD(t,true,true)>PayoffD(t,true,false);
foreach(var t in targets){Console.WriteLine($"{t}: réaction={Reaction(t)}, interception={PayoffD(t,true,true):F2}, sans réaction={PayoffD(t,true,false):F2}");foreach(var (covered,react) in new[]{(false,false),(true,false),(true,true)}) Console.WriteLine($"  couvert={covered}, réaction={react} : U_A={PayoffA(t,covered,react):F2}, U_D={PayoffD(t,covered,react):F2}");}

T0: réaction=False, interception=-0.50, sans réaction=0.50


  couvert=False, réaction=False : U_A=3.00, U_D=-1.00


  couvert=True, réaction=False : U_A=2.00, U_D=0.50


  couvert=True, réaction=True : U_A=0.00, U_D=-0.50


T1: réaction=True, interception=-0.50, sans réaction=-1.00


  couvert=False, réaction=False : U_A=4.00, U_D=-2.50


... (2 outputs left out)

T2: réaction=True, interception=-0.50, sans réaction=-2.50


  couvert=False, réaction=False : U_A=2.00, U_D=-4.00


  couvert=True, réaction=False : U_A=1.00, U_D=-2.50


  couvert=True, réaction=True : U_A=-1.00, U_D=-0.50


### Lecture

Pour T0, réagir donne −0,50 alors que laisser passer une attaque couverte donne +0,50 : la réaction n'est pas rationnelle. Pour T1 et T2, l'interception donne −0,50 et améliore le gain du défenseur ; les scénarios affichés distinguent couverture physique et interception.

## 2. Solveur exact — énumération pour 3 cibles

**Séquence du jeu** (l'ordre détermine qui décide quoi) :
1. le défenseur **s'engage** sur une allocation `x ⊆ T`, `|x| ≤ R` (observable — commit, comme GT-9b) ;
2. l'attaquant observe `x` et choisit `t` ;
3. si `t` est couverte : le capteur sonne avec probabilité `p_d` (sinon, faux négatif, rien) ;
4. **sur signal**, le défenseur choisit de réagir ou non — c'est **sa** décision, cible par cible (politique calculée en §1).

**Espérances** : l'attaquant anticipe la politique de réaction du défenseur et évalue
`E[U(t | x)] = p_d · U(t | signal, politique de réaction) + p_fn · U(t | pas de signal)`.
Sur cible non couverte, pas de capteur ni de réaction : paiements bruts.

**Départage Strong Stackelberg — codé, pas déclaré.** L'attaquant choisit `t* = argmax_t U_A(t | x)` ; **à ex æquo**, il prend la cible qui **maximise `U_D`** (le leader peut induire ce choix — c'est précisément ce qui distingue le Strong du Weak Stackelberg). Dans le solveur : `max(ex_aequos, key=U_D)`. Ce départage n'est pas décoratif : les tables ci-dessous contiennent des ex æquos **réels**, dont un triple à `p_fn = 0,5` — sans lui, le choix reviendrait à l'ordre de déclaration des cibles.

**Les 4 stratégies** = (croyance sur le capteur, politique de réaction), chacune **jouée puis évaluée en espérance réelle** (`p_d = 0,7`) pour être comparable :

| Stratégie | Croyance | Politique de réaction |
|-----------|----------|----------------------|
| **NO-RES** | pas de capteur | n/a (rien déployé) |
| **DET-SDP** (Tambe 2011) | capteur parfait (`p_d = 1`) | optimale sous cette croyance |
| **UNC-PESS** | `p_d` réel | réaction **désactivée** (supposée ineffective) |
| **UNC-OPT** | `p_d` réel | optimale (celle du §1) |

Pour la petite instance (3 cibles), on **énumère** les allocations `x` (`C(3,0) + C(3,1) + C(3,2) = 7` cas) et le défenseur maximise `U_D` **selon sa croyance** ; chaque ligne de la table finale rapporte l'espérance **réelle** obtenue en jouant la stratégie choisie.

In [4]:
// Mélanger détection et faux négatif seulement pour une cible couverte ; l'allocation vide reste admissible.
(double ua, double ud) Esperances(string t, int allocation, double pd, bool? reactPolicy = null)
{
    int i = Array.IndexOf(targets, t);
    if ((allocation & (1 << i)) == 0)
        return (va[t], vd[t]);
    bool react = reactPolicy ?? Reaction(t);
    return (pd * PayoffA(t, true, react) + (1 - pd) * PayoffA(t, true, false), pd * PayoffD(t, true, react) + (1 - pd) * PayoffD(t, true, false));
}

// Le départage SSE favorise le défenseur parmi les cibles ex aequo pour l'attaquant.
(string cible, double ua, double ud, int ties) BestResponse(int allocation, double pd, bool? reactPolicy = null)
{
    var cands = targets.Select(t =>
    {
        var u = Esperances(t, allocation, pd, reactPolicy);
        return (cible: t, ua: u.ua, ud: u.ud);
    }).ToArray();
    double best = cands.Max(x => x.ua);
    var ties = cands.Where(x => Math.Abs(x.ua - best) < 1e-12).ToArray();
    var choix = ties.OrderByDescending(x => x.ud).First();
    return (choix.cible, choix.ua, choix.ud, ties.Length);
}

// Énumérer tous les masques admissibles, en commençant par les allocations de plus petite cardinalité.
IEnumerable<int> Allocations() => Enumerable.Range(0, 8).Where(m => System.Numerics.BitOperations.PopCount((uint)m) <= R).OrderBy(m => System.Numerics.BitOperations.PopCount((uint)m)).ThenBy(m => m);
string NomAlloc(int m) => "[" + string.Join(", ", targets.Where((t, i) => (m & (1 << i)) != 0)) + "]";
record Resultat(string Strategie, int Alloc, string Cible, double UA, double UD, double UDBelief, int Ties);
// Optimiser sous la croyance puis évaluer sous pD réel ; ne pas confondre ces deux utilités.
Resultat Resoudre(string nom, double croyance, bool? reactPolicy = null, bool noDeploy = false)
{
    Resultat best = null;
    foreach (int a in noDeploy ? new[]
    {
        0
    }

    : Allocations())
    {
        var b = BestResponse(a, croyance, reactPolicy);
        var real = BestResponse(a, pD, reactPolicy);
        if (best == null || b.ud > best.UDBelief)
            best = new Resultat(nom, a, b.cible, real.ua, real.ud, b.ud, b.ties);
    }

    return best;
}

var results = new Dictionary<string, Resultat>
{
    {
        "NO-RES",
        Resoudre("NO-RES", pD, noDeploy: true)
    },
    {
        "DET-SDP",
        Resoudre("DET-SDP", 1)
    },
    {
        "UNC-PESS",
        Resoudre("UNC-PESS", pD, false)
    },
    {
        "UNC-OPT",
        Resoudre("UNC-OPT", pD)
    }
};
Console.WriteLine($"Quatre stratégies : p_d réel={pD:F2}");
foreach (var(nom, r)in results)
    Console.WriteLine($"{nom, -9} allocation={NomAlloc(r.Alloc), -10} cible croyance={r.Cible}, ex æquo={r.Ties}, U_A={r.UA:F2}, U_D={r.UD:F2}");


Quatre stratégies : p_d réel=0.70


NO-RES    allocation=[]         cible croyance=T1, ex æquo=1, U_A=4.00, U_D=-2.50


DET-SDP   allocation=[T0, T1]   cible croyance=T0, ex æquo=2, U_A=2.00, U_D=0.50


UNC-PESS  allocation=[T1]       cible croyance=T0, ex æquo=2, U_A=3.00, U_D=-1.00


UNC-OPT   allocation=[T0, T1]   cible croyance=T0, ex æquo=2, U_A=2.00, U_D=0.50


### Lecture : qui gagne, qui perd, et pourquoi

Comparer les `U_A` (utilité de l'attaquant, à **minimiser** du point de vue du défenseur) et les `U_D` (utilité du défenseur, à **maximiser**) — les deux sont rapportées en **espérance réelle** sur `p_d = 0,7`, chaque stratégie étant *choisie* sous sa croyance puis *évaluée* au vrai capteur.

- **NO-RES** : sans patrouille, l'attaquant prend sa cible préférée T1. U_A = +4,00, U_D = −2,50 — le pire résultat défenseur, c'est la référence du §3.
- **DET-SDP** : croyance capteur parfait (p_d = 1). Elle sélectionne [T0, T1] — qui se révèle être **aussi** le choix de UNC-OPT : la coïncidence DET-SDP = UNC-OPT est un fait mesuré de cette instance, pas une loi (elle tient sur toute la grille p_fn du §3).
- **UNC-PESS** : réaction désarmée par hypothèse. La pénalité restante vient de la couverture physique, indépendamment du capteur ; l'attaquant bascule vers T0 non couverte. U_D = −1,00 : désarmer la réaction coûte 1,50 d'utilité défenseur par rapport à UNC-OPT.
- **UNC-OPT** : [T0, T1] avec la politique de réaction rationnelle. L'attaquant est **ex æquo** entre T0 couverte (E[U_A] = +2,00 — pas de réaction rationnelle sur T0, trop peu coûteuse à perdre) et T2 non couverte (U_A = +2,00). Le départage SSE tranche vers T0 : U_D = +0,50 au lieu de −4,00 si l'attaquant frappait T2.

### Exercice 1 — Faire varier les ressources du défenseur

Complétez `strategiePourRessources(nb)` : changez temporairement `R`, résolvez UNC-OPT avec `Resoudre("UNC-OPT",pD)` puis restaurez `R` avec `try/finally`. Le résultat structuré porte `Alloc`, `Cible`, `UA` et `UD`. Testez 0, 1, 2 et 3 ressources et expliquez pourquoi les allocations admissibles s'élargissent.

In [5]:
Func<int,Resultat> strategiePourRessources=nb=>{
 // TODO : résoudre UNC-OPT et restaurer R, même en cas d'erreur
 return null;
};
int rInitial=R;var repR=new Resultat[4];for(int k=0;k<4;k++){repR[k]=strategiePourRessources(k);if(R!=rInitial)throw new Exception("R non restauré après appel");}
if(repR.All(x=>x==null))Console.WriteLine("Exercice 1 à compléter");
else{if(repR.Any(x=>x==null))throw new Exception("Réponse partielle");if(R!=rInitial)throw new Exception("R non restauré");for(int k=0;k<4;k++){Resultat attendu;try{R=k;attendu=Resoudre("UNC-OPT",pD);}finally{R=rInitial;}var x=repR[k];if(x.Alloc!=attendu.Alloc||x.Cible!=attendu.Cible||!double.IsFinite(x.UA)||!double.IsFinite(x.UD)||Math.Abs(x.UA-attendu.UA)>1e-9||Math.Abs(x.UD-attendu.UD)>1e-9)throw new Exception("Sonde incorrecte");}Console.WriteLine("Exercice 1 : allocations et restauration validées");}

Exercice 1 à compléter


## 3. Sweep du taux de faux négatifs

On fait varier $p_{fn}$ de 0 à 0,9, et chaque stratégie est évaluée avec le capteur réel du point.

In [6]:
var sweep=new List<(double fn,Resultat opt,Resultat det,Resultat pess,Resultat nores,bool bondi)>();
double pDSave=pD;
for(int k=0;k<10;k++){double fn=k/10.0;pD=1-fn;var opt=Resoudre("UNC-OPT",pD);var det=Resoudre("DET-SDP",1);var pess=Resoudre("UNC-PESS",pD,false);var nores=Resoudre("NO-RES",pD,noDeploy:true);sweep.Add((fn,opt,det,pess,nores,opt.UD<nores.UD-1e-12));}
pD=pDSave;
foreach(var s in sweep)Console.WriteLine($"p_fn={s.fn:F2} alloc={NomAlloc(s.opt.Alloc),-9} cible={s.opt.Cible} ex æquo={s.opt.Ties} UA={s.opt.UA:F2} UD={s.opt.UD:F2} NO-RES={s.nores.UD:F2} critère={s.bondi}");
Console.WriteLine($"Critère Bondi : {sweep.Count(x=>x.bondi)}/{sweep.Count}");

p_fn=0.00 alloc=[T0, T1]  cible=T0 ex æquo=2 UA=2.00 UD=0.50 NO-RES=-2.50 critère=False


p_fn=0.10 alloc=[T0, T1]  cible=T0 ex æquo=2 UA=2.00 UD=0.50 NO-RES=-2.50 critère=False


p_fn=0.20 alloc=[T0, T1]  cible=T0 ex æquo=2 UA=2.00 UD=0.50 NO-RES=-2.50 critère=False


p_fn=0.30 alloc=[T0, T1]  cible=T0 ex æquo=2 UA=2.00 UD=0.50 NO-RES=-2.50 critère=False


p_fn=0.40 alloc=[T0, T1]  cible=T0 ex æquo=2 UA=2.00 UD=0.50 NO-RES=-2.50 critère=False


p_fn=0.50 alloc=[T0, T1]  cible=T0 ex æquo=3 UA=2.00 UD=0.50 NO-RES=-2.50 critère=False


p_fn=0.60 alloc=[T0, T1]  cible=T1 ex æquo=1 UA=2.20 UD=-0.80 NO-RES=-2.50 critère=False


p_fn=0.70 alloc=[T0, T1]  cible=T1 ex æquo=1 UA=2.40 UD=-0.85 NO-RES=-2.50 critère=False


p_fn=0.80 alloc=[T0, T1]  cible=T1 ex æquo=1 UA=2.60 UD=-0.90 NO-RES=-2.50 critère=False


p_fn=0.90 alloc=[T0, T1]  cible=T1 ex æquo=1 UA=2.80 UD=-0.95 NO-RES=-2.50 critère=False


Critère Bondi : 0/10


### Lecture : transition de phase à p_fn = 0,5 — et pourquoi Bondi ne se déclenche pas

Deux phénomènes lisibles dans la table :

1. **Transition de phase à `p_fn = 0,5`.** Tant que le capteur détecte plus d'une fois sur deux, l'attaquant est ex æquo entre T0 couverte et T2 non couverte (U_A = +2,00) — et à `p_fn = 0,5` exactement, les **trois** cibles sont ex æquo : le départage SSE (max U_D) est alors le seul mécanisme qui décide où frappe l'attaquant, et il choisit T0 (U_D = +0,50). Dès `p_fn > 0,5`, T1 couverte redevient le meilleur coup de l'attaquant — la protection d'espérance de la patrouille fond avec `p_d`, elle ne dissuade plus — et U_D glisse de +0,50 à −0,95. Le capteur dégradé fait ré-entrer l'attaquant sur la cible chère : la table ne peut plus être constante, la structure du jeu traverse une frontière.

2. **Le critère Bondi ne se déclenche jamais (0/10) — cause structurelle, pas un hasard de chiffres.** Dans ce modèle, UNC-OPT maximise le gain du défenseur sur un ensemble d'allocations qui contient l'allocation vide : il peut donc toujours choisir le résultat NO-RES. L'absence de faux positifs supprime en outre un coût d'alerte sans attaque ; elle ne prouve pas à elle seule un théorème pour toute autre politique ou tout autre modèle. Le phénomène de Bondi demanderait une autre formulation, avec les hypothèses correspondantes.

### Exercice 2 — Engagement contre un mélange uniforme

Complétez `evaluerMelange(pd)` en répartissant uniformément les allocations de taille exactement `R`. Calculez l'utilité moyenne par cible, la meilleure réponse SSE et l'utilité correspondante du défenseur ; retournez un `Melange`. Comparez avec UNC-OPT : l'attaquant réagit aux moyennes du mélange, pas à chaque allocation individuellement.

In [7]:
record Melange(Dictionary<string,double> UAParCible,string Cible,double UA,double UD);
Func<double,Melange> evaluerMelange=pd=>{
 // TODO : moyennes par cible, puis départage SSE
 return null;
};
var casMelange=new[]{(0.0,new[]{7.0/3,10.0/3,4.0/3},"T1",-1.5),(0.7,new[]{7.0/3,2.4,.4},"T1",-19.0/15),(1.0,new[]{7.0/3,2.0,0.0},"T0",0.0)};
var repMelange=casMelange.Select(x=>evaluerMelange(x.Item1)).ToArray();
if(repMelange.All(x=>x==null))Console.WriteLine("Exercice 2 à compléter");
else{if(repMelange.Any(x=>x==null))throw new Exception("Réponse partielle");for(int i=0;i<casMelange.Length;i++){var x=repMelange[i];var t=casMelange[i];if(x.UAParCible==null||x.UAParCible.Count!=3||targets.Any(z=>!x.UAParCible.ContainsKey(z)))throw new Exception("Clés incorrectes");for(int j=0;j<3;j++)if(!double.IsFinite(x.UAParCible[targets[j]])||Math.Abs(x.UAParCible[targets[j]]-t.Item2[j])>1e-9)throw new Exception("Moyenne incorrecte");if(x.Cible!=t.Item3||!double.IsFinite(x.UA)||!double.IsFinite(x.UD)||Math.Abs(x.UA-t.Item2.Max())>1e-9||Math.Abs(x.UD-t.Item4)>1e-9)throw new Exception("Sonde incorrecte");}Console.WriteLine($"Exercice 2 : sondes validées ; comparer à UNC-OPT {results["UNC-OPT"].UD:F2}");}

Exercice 2 à compléter


## 4. Sensibilité aux paiements attaquant

On fait varier la valeur de T1 parmi 2, 4 et 6, puis on compare le déploiement optimal à NO-RES.

In [8]:
var sensibilite=new List<(double valeur,Resultat opt,Resultat nores)>();
foreach(double valeur in new[]{2.0,4.0,6.0}){double ancien=va["T1"];try{va["T1"]=valeur;sensibilite.Add((valeur,Resoudre("UNC-OPT",pD),Resoudre("NO-RES",pD,noDeploy:true)));}finally{va["T1"]=ancien;}}
foreach(var s in sensibilite)Console.WriteLine($"V_A[T1]={s.valeur:F1}, alloc={NomAlloc(s.opt.Alloc)}, cible={s.opt.Cible}, ex æquo={s.opt.Ties}, U_A={s.opt.UA:F2}, U_D={s.opt.UD:F2}, NO-RES U_A={s.nores.UA:F2}, U_D={s.nores.UD:F2}, aide={s.opt.UD>s.nores.UD}");

V_A[T1]=2.0, alloc=[T0], cible=T0, ex æquo=3, U_A=2.00, U_D=0.50, NO-RES U_A=3.00, U_D=-1.00, aide=True


V_A[T1]=4.0, alloc=[T0, T1], cible=T0, ex æquo=2, U_A=2.00, U_D=0.50, NO-RES U_A=4.00, U_D=-2.50, aide=True


V_A[T1]=6.0, alloc=[T1], cible=T1, ex æquo=1, U_A=3.60, U_D=-0.65, NO-RES U_A=6.00, U_D=-2.50, aide=True


### Lecture

La table mesure l'effet sur les paiements du défenseur et le nombre d'ex æquos ; lorsque la valeur de T1 vaut 2, une seule patrouille et le départage SSE suffisent dans cette instance.

### Exercice 3 — Condition d'indifférence de l'attaquant

Complétez `ciblesIndifferentes(alloc,pd,tol)` avec les cibles à `tol` près du meilleur gain attaquant. L'allocation est un masque de bits : 1, 2, 4 désignent T0, T1, T2 ; le masque 3 couvre T0 et T1. Comparez les ex æquos à la sortie du solveur, puis baissez légèrement la valeur de T2 et observez l'effet sans modifier durablement l'instance.

In [9]:
Func<int,double,double,List<string>> ciblesIndifferentes=(alloc,pd,tol)=>{
 // TODO : cibles atteignant le maximum attaquant à tol près
 return null;
};
var casIndiff=new[]{(0,.7,new[]{"T1"}),(3,.7,new[]{"T0","T2"}),(3,.5,new[]{"T0","T1","T2"}),(2,.7,new[]{"T0"})};
var repIndiff=casIndiff.Select(x=>ciblesIndifferentes(x.Item1,x.Item2,1e-9)).ToArray();
if(repIndiff.All(x=>x==null))Console.WriteLine("Exercice 3 à compléter");
else{if(repIndiff.Any(x=>x==null))throw new Exception("Réponse partielle");for(int i=0;i<casIndiff.Length;i++)if(repIndiff[i].Count!=casIndiff[i].Item3.Length||!repIndiff[i].OrderBy(x=>x).SequenceEqual(casIndiff[i].Item3.OrderBy(x=>x)))throw new Exception("Sonde incorrecte");double ancien=va["T2"];try{va["T2"]=ancien-.1;var perturbe=ciblesIndifferentes(3,.7,1e-9);if(perturbe==null||!perturbe.SequenceEqual(new[]{"T0"}))throw new Exception("Perturbation incorrecte");}finally{va["T2"]=ancien;}Console.WriteLine("Exercice 3 : sondes et perturbation validées");}

Exercice 3 à compléter


## 5. Verdict final -- resume et limites

Tableau de synthèse (espérances réelles, `p_d = 0,7`) :

| Stratégie | Croyance capteur | Allocation | t* | U_A | U_D |
|-----------|------------------|------------|----|-----|-----|
| NO-RES | — | ∅ | T1 | +4,00 | −2,50 |
| DET-SDP (Tambe 2011) | parfait (`p_d = 1`) | [T0, T1] | T0 (départage) | +2,00 | **+0,50** |
| UNC-PESS | réel, réaction off | [T1] | T0 (départage) | +3,00 | −1,00 |
| UNC-OPT | réel (`p_d = 0,7`) | [T0, T1] | T0 (départage) | +2,00 | **+0,50** |

Trois lectures :

- **Le départage SSE est porteur de valeur.** À [T0, T1], l'attaquant est ex æquo entre T0 (couverte, E[U_A] = +2,00) et T2 (non couverte, +2,00). Sans départage favorable, il pourrait frapper T2 (U_D = −4,00) ; avec, il frappe T0 (U_D = +0,50). L'hypothèse « l'attaquant tranche les ex æquo en faveur du leader » vaut ici **4,50 d'utilité défenseur** — c'est la part du résultat qui vient de la *théorie* du Strong Stackelberg, pas des paiements.
- **DET-SDP = UNC-OPT sur cette instance** : la croyance capteur-parfait sélectionne la même allocation que la croyance réaliste, et la coïncidence tient sur toute la grille `p_fn` (§3). Fait mesuré de l'instance, pas une loi — sur d'autres paiements, DET-SDP sur-investit dans la réaction que le vrai capteur ne permet pas.
- **Ce que ce notebook NE montre PAS** : le résultat « capteur strictement moins bon que rien » (Bondi et al. 2019) est **absent** (0/10 au critère U_D) par construction pour UNC-OPT : l'allocation vide reste admissible, donc son optimum est au moins égal à NO-RES. L'absence de faux positifs supprime également les réactions sans attaque ; elle ne suffit pas à établir le même résultat pour un autre modèle ou une politique contrainte. L'énumération reste possible pour `R=3` et quelques cibles ; quand l'espace combinatoire devient trop grand, un solveur MILP peut être envisagé selon le budget. PuLP est seulement un import optionnel de la source, jamais invoqué par ses expériences.

**Raccord GT-9b** : on a prolongé l'engagement crédible en déploiement **sur un graphe** sous incertitude. Le défenseur s'engage sur **toute** son allocation (commit), le follower observe avant d'attaquer, et la valeur du leader vient maintenant de deux sources : la couverture physique **et** la discipline de l'attaquant sur les ex æquo (départage SSE).

## 6. Synthèse finale

Récapitulons les quatre politiques, la transition à $p_{fn}=0,5$ et les trois valeurs de T1.

In [10]:
var rBase=results["UNC-OPT"];
Console.WriteLine($"Instance : 3 cibles, R={R}, p_d={pD:F2}, p_fn={1-pD:F2}, p_fp=0");
Console.WriteLine($"UNC-OPT : {NomAlloc(rBase.Alloc)}, cible={rBase.Cible}, {rBase.Ties} ex æquo, U_A={rBase.UA:F2}, U_D={rBase.UD:F2}, NO-RES={results["NO-RES"].UD:F2}");
foreach(var s in sweep)Console.WriteLine($"p_fn={s.fn:F2}, cible={s.opt.Cible}, U_D={s.opt.UD:F2}, ex æquo={s.opt.Ties}");
Console.WriteLine($"Critère Bondi : {sweep.Count(x=>x.bondi)}/{sweep.Count} ; sensibilité : amélioration dans {sensibilite.Count(s=>s.opt.UD>s.nores.UD)}/{sensibilite.Count} cas.");

Instance : 3 cibles, R=2, p_d=0.70, p_fn=0.30, p_fp=0


UNC-OPT : [T0, T1], cible=T0, 2 ex æquo, U_A=2.00, U_D=0.50, NO-RES=-2.50


p_fn=0.00, cible=T0, U_D=0.50, ex æquo=2


p_fn=0.10, cible=T0, U_D=0.50, ex æquo=2


p_fn=0.20, cible=T0, U_D=0.50, ex æquo=2


p_fn=0.30, cible=T0, U_D=0.50, ex æquo=2


... (3 outputs left out)

p_fn=0.70, cible=T1, U_D=-0.85, ex æquo=1


p_fn=0.80, cible=T1, U_D=-0.90, ex æquo=1


p_fn=0.90, cible=T1, U_D=-0.95, ex æquo=1


Critère Bondi : 0/10 ; sensibilité : amélioration dans 3/3 cas.


## À retenir

L'allocation, la réaction rationnelle et le départage SSE déterminent conjointement les gains. L'allocation vide est admissible : l'optimum calculé ne peut être inférieur au gain NO-RES dans ce modèle.

## Vérifiez votre compréhension

1. Qui décide de réagir au signal ?

<details><summary>Réponse</summary>Le défenseur compare son gain d'interception à son gain sans réaction, cible par cible.</details>

2. Pourquoi un ex æquo de l'attaquant compte-t-il ?

<details><summary>Réponse</summary>Le départage SSE choisit, parmi les meilleures réponses attaquant, celle qui maximise le gain du défenseur.</details>

## Pour aller plus loin

- [Engagement Stackelberg](GameTheory-09b-Commitment-Stackelberg-Python.ipynb).
- Tambe, *Security and Game Theory*, Cambridge University Press, 2011.